# ДЗ: очистка даних та рейтинг заявок

Цей ноутбук:
1) Завантажує `applications.csv` та чистить дані.
2) Додає до заявок рейтинги індустрій з `industries.csv`.
3) Рахує рейтинг заявки (0–100) за заданими правилами.
4) Залишає лише прийняті заявки (rating > 0).
5) Групує за тижнем подачі та рахує середній рейтинг по тижнях.

In [ ]:
import pandas as pd
import numpy as np

# Шляхи до файлів (у цій задачі файли вже завантажені в середовище)
applications_path = r"/mnt/data/applications(2.0).csv"
industries_path = r"/mnt/data/industries(2.0).csv"


In [ ]:
# 1) Завантажуємо applications.csv
df = pd.read_csv(applications_path)
display(df.head())
print("Rows:", len(df), "| Cols:", df.shape[1])


In [ ]:
# 1) Чистка даних

# 1.1 Прибираємо дублікати applicant_id
before = len(df)
df = df.drop_duplicates(subset=["applicant_id"], keep="first").copy()
after = len(df)
print(f"Дублікати applicant_id видалено: {before-after} (залишилось {after})")

# 1.2 External Rating: заповнюємо пропуски 0
df["External Rating"] = df["External Rating"].fillna(0)

# 1.3 Education level: заповнюємо пропуски 'Середня'
df["Education level"] = df["Education level"].fillna("Середня")

# Перевірка
print("External Rating NA:", df["External Rating"].isna().sum())
print("Education level NA:", df["Education level"].isna().sum())


In [ ]:
# 2) Додаємо рейтинги індустрій з industries.csv

industries = pd.read_csv(industries_path)
display(industries.head())

# Об'єднання по полю Industry
df = df.merge(industries, on="Industry", how="left")

# Якщо для якоїсь індустрії немає Score — вважаємо 0
df["Score"] = df["Score"].fillna(0)

display(df.head())


In [ ]:
# 3) Розрахунок рейтингу заявки (0–100)

# Дата/час подачі (формат у файлі: '11.30.2022 10:26:37' => %m.%d.%Y %H:%M:%S)
df["Applied at"] = pd.to_datetime(df["Applied at"], format="%m.%d.%Y %H:%M:%S", errors="coerce")

# 6 критеріїв:
age_points = df["Age"].between(35, 55, inclusive="both") * 20

# Не вихідні: Пн-Пт (weekday 0-4)
not_weekend_points = (df["Applied at"].dt.weekday < 5) * 20

married_points = (df["Marital status"] == "Married") * 20

kyiv_points = (df["Location"] == "Київ чи область") * 10

industry_points = df["Score"].clip(lower=0, upper=20)  # 0..20

ext_bonus = (df["External Rating"] >= 7) * 20
ext_penalty = (df["External Rating"] <= 2) * 20
external_points = ext_bonus - ext_penalty  # може бути -20, 0 або +20

# Сума балів
df["rating_raw"] = (
    age_points
    + not_weekend_points
    + married_points
    + kyiv_points
    + industry_points
    + external_points
)

# Умова "rating = 0", якщо Amount відсутній або External Rating == 0
mask_zero = df["Amount"].isna() | (df["External Rating"] == 0)
df["rating"] = df["rating_raw"].where(~mask_zero, 0)

# Обмежуємо до 0..100
df["rating"] = df["rating"].clip(lower=0, upper=100).astype(int)

display(df[["applicant_id","Applied at","Amount","Age","Industry","Score","External Rating","Marital status","Location","rating_raw","rating"]].head(10))
print("Min rating:", df["rating"].min(), "| Max rating:", df["rating"].max())


In [ ]:
# 4) Залишаємо лише прийняті заявки (rating > 0)

accepted = df[df["rating"] > 0].copy()
print("Прийняті заявки:", len(accepted), "із", len(df))

display(accepted.head())


In [ ]:
# 5) Групування за тижнем подачі та середній рейтинг прийнятих заявок

# Тиждень як початок тижня (понеділок) для зручного читання
accepted["week_start"] = accepted["Applied at"].dt.to_period("W").apply(lambda p: p.start_time)

weekly_avg = (
    accepted
    .groupby("week_start", as_index=False)["rating"]
    .mean()
    .rename(columns={"rating":"avg_rating"})
    .sort_values("week_start")
)

display(weekly_avg)
